# **Planteamiento inicial para el proyecto**

problmea - que buscaba hacer - que ha salido - razonamiento 


# **Problemas y resoluciones durante el proyecto**

#### Etiquetas ID de tipos de problemas

##### Categorías principales

| Categoría principal | Etiqueta   | Qué significa                                                                 |
|--------------------|-----------|------------------------------------------------------------------------------|
| Environment        | [ENV]     | Entorno Python, dependencias, rutas, configuración local                    |
| Version Control    | [VCS]     | Git, GitHub, control de versiones                                           |
| HTTP / Networking  | [HTTP]    | Llamadas a APIs: códigos de estado, rate limits, autenticación              |
| Authentication     | [AUTH]    | API keys, tokens, permisos                                                  |
| Data Quality       | [DATA]    | Calidad, formato o estructura de datos extraídos                            |
| Configuration      | [CONFIG]  | Archivos de configuración, variables de entorno, secrets                    |
| Machine Learning   | [ML]      | Modelos, features, entrenamiento, métricas                                  |
| EDA                | [EDA]     | Visualizaciones, estadísticas, distribuciones, outliers                     |
| Web Scraping       | [SCRAPING]| Extracción de datos web: HTML, selectores, parsing                          |
| Jupyter Notebook   | [NOTEBOOK]| Errores específicos de ejecución o estructura                               |

##### Subcategorías

| Subcategoría        | Etiqueta         | Qué significa                                                                 |
|--------------------|------------------|------------------------------------------------------------------------------|
| Dependencies       | [DEPS]           | Dependencias Python no instaladas o con versión incorrecta                  |
| Rate Limit         | [RATE-LIMIT]     | Servidor bloqueó peticiones por exceso de velocidad                         |
| Timeout            | [TIMEOUT]        | La petición tardó demasiado y fue cancelada                                 |
| Missing Data       | [MISSING-DATA]   | Valores nulos, campos vacíos, datos incompletos                             |
| Schema Mismatch    | [SCHEMA]         | Estructura de respuesta de API no coincide con lo esperado                  |
| Secrets Management | [SECRETS]        | Gestión de claves y credenciales sensibles                                  |
| Batch Processing   | [BATCH]          | Procesamiento en lote vs peticiones individuales                            |
| Pagination         | [PAGINATION]     | Manejo de resultados paginados en APIs                                      |
| Feature Engineering| [FEATURE-ENG]    | Creación y transformación de variables para ML                              |
| Coverage           | [COVERAGE]       | Baja cobertura de datos entre fuentes                                       |
| Deprecation        | [DEPRECATION]    | API o servicio discontinuado o sin mantenimiento                            |
| Silent Failure     | [SILENT-FAIL]    | Fallo sin error visible, devuelve datos incorrectos silenciosamente         |

### Fase 1 — Extracción de datos con la API de Last.fm



> Los tres endpoints utilizados

| **Endpoint** | **Qué devuelve** | **Para qué se usa** |
|---|---|---|
| `tag.getTopTags` | 50 géneros globales con name, count, reach | Construir `df_tags` |
| `tag.getTopTracks` | Tracks por género (hasta 1.000 por tag) | Descubrir mbids de canciones |
| `track.getInfo` | Metadata completa por mbid | Genera `backup_tracks.csv` |

**Por qué Last.fm:** Devuelve datos basados en consumo real de usuarios — no estimaciones. Las métricas de `playcount` y `listeners` son acumuladas históricas reales.

> Los tres CSV del proyecto

| **CSV** | **Origen** | **Columnas únicas** | **Filas** |
|---|---|---|---|
| `backup_tracks.csv` | `track.getInfo` | tag (géneros), published (fecha), duration (ms) | 34.417 |
| `lastfm_dataset.csv` | Pipeline multi-endpoint | country, genre_tag, rank_global, rank_by_country | 59.999 |
| `tags_dataset.csv` | `tag.getTopTags` | name, count, reach — los 50 tags más populares | 50 |

**Nota sobre la columna `tag`:** Last.fm devuelve los géneros como lista de diccionarios asignados por usuarios de la plataforma: `"[{'name': 'rock', 'url': '...'}, ...]"`. Solo el **10.5%** de los tracks tiene tag con contenido — Last.fm solo los devuelve cuando los usuarios han etiquetado manualmente esa canción.

> **ERROR 1:**  
> HTTPError: 429 Too Many Requests  
> Response: {'error': 29, 'message': 'Rate limit exceeded'}

**Etiquetas:** `[HTTP]` `[RATE-LIMIT]` `[BATCH]`

**Contexto:**  
Para alcanzar el volumen necesario de tracks hay que hacer cientos de llamadas a la API. Last.fm limita las peticiones por API key a aproximadamente **5 peticiones por segundo** con el plan gratuito. Un bucle sin control de velocidad dispara el rate limit inmediatamente.

**¿Qué es un Rate Limit?**  
Un rate limit es una restricción que limita cuántas peticiones puede hacer un cliente en un período de tiempo determinado. Se expresa como X peticiones por segundo/minuto/hora. Cuando se supera, el servidor responde con **HTTP 429 Too Many Requests**.

**Solución — Throttling con `time.sleep()`:** Implemento throttling explícito con `time.sleep()` respetando el límite declarado por la API con un margen de seguridad. Para proyectos grandes uso retry con backoff exponencial ante 429, guardo resultados parciales en disco para tolerar interrupciones, y diversifico los endpoints para distribuir la carga.


#### AcousticBrainz: API discontinuada (~98% NaN)

> **ERROR 2:**  
> Se intentó enriquecer el dataset con metadata de audio (BPM, danceability, key, mood) desde la API de AcousticBrainz. Al analizar una muestra de 500 tracks: **~98% de valores NaN** en todas las columnas excepto `mbid`.

**Etiquetas:** `[DATA]` `[MISSING-DATA]` `[DEPRECATION]`

**Causa:**  
AcousticBrainz dejó de actualizarse y fue discontinuada antes que Last.fm. La mayoría de canciones actuales del dataset no tienen metadata en esa base de datos.

**Resolución:**  
Se documenta como limitación del proyecto. Las predicciones del modelo se basan únicamente en métricas de popularidad de Last.fm, no en características acústicas.

**Alternativas evaluadas:**

| **Opción** | **Pros** | **Contras** |
|---|---|---|
| Spotify API (Audio Features) | Tiene danceability, energy, BPM, valence | Requiere cuenta de developer + OAuth |
| Dataset FMA (Free Music Archive) | Libre, 106.574 canciones con features | Difícil de cruzar con mbids de Last.fm |
| Mantener solo Last.fm | Ya disponible, sin extra | Sin metadata acústica |

**Decisión:** Se mantiene el scope con Last.fm. Las features acústicas (BPM, danceability) quedan documentadas como ampliación futura.

#### TRATANDO DATA DE LAS APIS

**· Problema 1:** Descarga de datos con API Lastfm: 
* Inicialmente se utilizan los endpoints "chart.getTopTracks","geo.getTopTracks", "tag.getTopTracks" y "track.getInfo".
* Se han hecho varias pruebas de descarga de datos con el objetivo de análisis de los tracks en el mercado musical según streams, popularidad y escuchas pero se ha encontrado data insuficiente con muchos valores faltantes, con respuestas muy generales o poco coherentes para el análisis.

**Resolución de la descarga de datos:** Búsqueda de nuevas apis para enfocar mejor el análisis y obtener más data. Se decide utilizar Acousticbrainz porque contiene la metadata de cada track. 

**· Problema 2:** Descarga de datos con Acousticbrainz:
* Se intenta recolectar data de los endpoints "Low-level " y "High-level" que devuelven la información del bpm, key, scale,danceability, mood_happy y genre_ab.
* Al analizar una muestra de 500 datos se oberva que existen nulos en todas las columnas menos en mbid. Esto indica que de las canciones que tenemos en el csv de Lastfm no hay data. Es posible que sea debido a que la misma base de datos dónde buscamos la metadata se dejó de actualizar antes que Lastfm."

**Resolución:** 
* Ideas, (1) utilizar solamente api acousticbrainz con data antigua com prueba... 

(2) intentar hacer web scrapping se paginas como Bandcamp o Nina Protocol.

Last.fm → popularidad (mainstream)

AcousticBrainz → características audio

Bandcamp → géneros + país

Nina → tendencias + escenas underground. Complementa a la data de LAst.fm porque añade la infromacion de las tendencias y el consumo mainstream de usuarios.

* Entre Bandcamp y Nina

| Feature        | Bandcamp | Nina                  |
| -------------- | -------- | --------------------- |
| Géneros        | ✅        | ⚠️ menos estructurado |
| País           | ✅        | ⚠️ depende            |
| Trends         | ⚠️       | ✅ MUY fuerte          |
| Underground    | ⚠️       | ✅ fuerte              |
| Datos abiertos | ❌        | ✅                     |




### Descripción de los problemas

* Primera prueba del proyecto - ipynb "Modulo 1_backup_manual_data":

**· Problema 1 & Resolución:** Identificamos que no es viable hacer focus en tres modulos así que reducimos a dos: Análisis del mercado musical y Fraude en streams.


**· Problema 2:** Descarga de información de usuarios.
* Se planteó el problema de conseguir datos de más usuarios.

**Resolución de la descarga de información de usuarios:** Se define una función para buscarlos entre los amigos del usuario llamado "rj" creando así una red de usarios conectados.

**· Problema 3:** Al revisar el código y la información que devuelve se observa que:
* 

### Fase 2 — Merged de la data (df_merged)

> **ERROR crítico:**  
> "Merge explosionado" : Cuando haces un JOIN entre dos tablas y una de ellas tiene duplicados en la columna clave (`mbid`), cada fila de la tabla izquierda se multiplica por el número de coincidencias en la tabla derecha. Con 15 duplicados por mbid, cada canción se convierte en 15 filas idénticas.
> El merge original se hacía sin deduplicar `lastfm_dataset`. Un mismo `mbid` aparece hasta **15 veces** en `lastfm_dataset` porque el pipeline lo recogía desde 15 endpoints distintos (chart global + 10 países + varios tags). El mismo track aparece con `country=GLOBAL`, `country=Spain`, `country=US`, etc.

**Contexto**

Un mismo `mbid` aparece hasta **15 veces** en `lastfm_dataset` porque el pipeline lo recogía desde 15 endpoints distintos (chart global + 10 países + varios tags). El mismo track aparece con `country=GLOBAL`, `country=Spain`, `country=US`, etc.

| **Versión del merge** | **Filas resultantes** | 
|---|---|---|
| Sin deduplicar (original) | 512.092 filas (×14.9) | 
| Con deduplicación por mbid | 34.341 filas | 

**Solución — Deduplicar con prioridad de country:**

* ¿Qué es un left join y cuándo puede generar duplicados?"*  

Un left join devuelve todas las filas de la tabla izquierda y las coincidencias de la derecha. Si la tabla derecha tiene múltiples filas para la misma clave, cada fila de la izquierda se multiplica. Para evitarlo hay que deduplicar la tabla derecha antes del join, eligiendo qué fila conservar con criterio de negocio.

**Cobertura real de la columna country**


| **Valor** | **Filas** | **% del total** | **Significado** |
|---|---|---|---|
| UNKNOWN | 12.720 | 37.0% | Tracks de `tag.getTopTracks` — sin país |
| GLOBAL | 1.874 | 5.5% | Tracks de `chart.getTopTracks` — sin país |
| País real | 1.979 | **5.8%** | Tracks de `geo.getTopTracks` — con país |
| Sin match (NaN) | 17.769 | 51.7% | mbid no está en lastfm_dataset |

**Conclusión:** `country` es útil para el EDA geográfico pero **no se incluye como feature del modelo** porque el 94% tiene valor no informativo.

### Fase 3 — Limpieza de datos

> **ERROR 1:**  
> `dropna()` sin efecto (no reasigna).

**Etiquetas:** `[DATA]` `[NOTEBOOK]`

**Síntoma:**  
`df_clean.dropna(subset=['name', 'artist'])` no modificaba `df_clean`.

**Causa:**  
En pandas, `dropna()` devuelve un **nuevo DataFrame** sin modificar el original. Si no reasignas el resultado, la variable no cambia.

**Solución:**
* Reasignar el resultado
df_clean = df_clean.dropna(subset=['name', 'artist'])

* En pandas, la mayoría de operaciones devuelven una copia nueva. Para modificar el DataFrame hay que reasignar el resultado o usar `inplace=True` (aunque se desaconseja en código moderno).

> **ERROR 2:** Columna `tag` mal estructurada

**Etiquetas:** `[DATA]` `[FEATURE-ENG]`

**Síntoma:**  
La columna `tag` no contenía el género como string sino como string con lista de diccionarios: `"[{'name': 'rock', 'url': '...'}, {'name': 'classic rock', 'url': '...'}]"`

**Causa:**  
Last.fm devuelve los tags como lista de objetos — múltiples géneros asignados por distintos usuarios. Al guardar en CSV se serializa como string.

**Solución:** Función `ast.literal_eval`

* ¿Qué es `ast.literal_eval`?  
La función convierte un string que representa una estructura Python (lista, diccionario) en el objeto Python correspondiente. Es más seguro que `eval()` porque solo acepta literales Python válidos, nunca ejecuta código.

> **ERROR 3:** — `is_short_track` con lógica invertida

**Etiquetas:** `[DATA]` `[ML]` `[NOTEBOOK]`

**Síntoma:**  
El modelo aprendía que "canción corta" = canción larga y viceversa. La feature importance de `is_short_track` daba resultados contrarios a lo esperado.

**Causa:**  
Una celda posterior sobreescribía la variable con la lógica contraria. 

**Solución:**  
Eliminar la celda 110 que revierte el filtro. La definición correcta es `< 2.5` — canciones más cortas que 2.5 minutos se consideran formato corto (TikTok/Reels).

**Resultado real:**
- Canciones cortas (< 2.5 min): **4.002 canciones (11.7%)**
- Canciones largas (≥ 2.5 min): **30.339 canciones (88.3%)**

## Fase 4 — Feature Engineering

### Variables derivadas creadas



| **#** | **Feature** | **Cómo se calcula** | **Por qué es útil** |
|---|---|---|---|
| 1 | `duration_min` | `duration / 60.000` | La API devuelve ms, necesitamos minutos |
| 2 | `is_short_track` | `1 si duration_min < 2.5` | Flag formato TikTok/Reels |
| 3 | `is_hit` | `1 si playcount ≥ p90` | **Target del modelo** — top 10% |
| 4 | `playcount_per_listener` | `playcount / listeners` | Engagement: cuántas veces escucha cada fan |
| 5 | `log_playcount` | `log1p(playcount)` | Elimina el sesgo de la distribución |
| 6 | `log_listeners` | `log1p(listeners)` | Idem para oyentes |
| 7 | `artist_track_count` | groupby artista, count | Artistas con más tracks suelen ser más conocidos |
| 8 | `track_share_of_artist` | `playcount / total_artista` | Si este track es el gran hit del artista |
| 9 | `tag_encoded` | LabelEncoder sobre `tag` | Convierte género texto → número para el modelo |

**¿Qué es una transformación logarítmica?**  
La distribución de `playcount` está muy sesgada — unas pocas canciones tienen cientos de millones de reproducciones y la mayoría tiene menos de un millón. Esto dificulta el aprendizaje del modelo. La transformación `log1p(x)` "comprime" los valores grandes y "estira" los pequeños, acercando la distribución a una forma normal y mejorando el rendimiento del modelo.

**Umbral de hit (p90):**  
Se define "hit" como estar en el **top 10% de reproducciones** del dataset.  
- Umbral calculado: `df_clean['playcount'].quantile(0.90)` = **5.079.698 plays**
- Hits: ~3.434 canciones (10%)
- No hits: ~30.907 canciones (90%)

* ¿Por qué usas el percentil 90 como umbral? 

Con p90 el modelo tiene un balance razonable: 10% hits vs 90% no hits. Con p75 habría demasiados hits (25%) y el modelo aprendería patrones menos discriminativos. Con p95 habría muy pocos para aprender bien la clase minoritaria.


## Fase 5 — EDA (Análisis Exploratorio)

### Hallazgos principales


**Distribución de reproducciones:**
- `playcount` tiene skewness muy alta (~15) → distribución exponencial, muy sesgada
- `log_playcount` tiene skewness ~0.3 → distribución casi normal
- Conclusión: necesario usar `log_playcount` como variable en el modelo

**Top artistas:**
- Los 5 artistas más populares concentran una fracción muy alta del total de reproducciones
- BTS lidera ampliamente con 3 canciones en el top 15 (Dynamite: 174M, Butter: 146M, FAKE LOVE: 128M)
- Mercado muy concentrado en pocos artistas mainstream

**Cobertura de géneros:**
- Solo el 10.5% de tracks tiene género asignado (3.590 de 34.341)
- Los géneros más populares por reproducciones totales: pop, rock, indie, alternative, Hip-Hop

### Tests estadísticos aplicados

| **Test** | **Pregunta** | **Tipo** | **Por qué no paramétrico** |
|---|---|---|---|
| Mann-Whitney U | ¿Canciones cortas tienen más plays que largas? | No paramétrico | Playcount no sigue distribución normal |
| Kruskal-Wallis | ¿El género influye en la popularidad? | No paramétrico | Idem |
| Spearman | ¿Correlación entre duración y popularidad? | No paramétrico | Variables con distribución sesgada |

**¿Cuándo usar tests no paramétricos?**  
Los tests paramétricos (t-test, ANOVA) asumen que los datos siguen una distribución normal. Cuando los datos están muy sesgados — como `playcount` — se usan alternativas no paramétricas que no hacen ese supuesto y son más robustas con distribuciones asimétricas.

## Fase 6 — Modelado


**Nota sobre el modelo guardado:**  
El `modelo_hit.pkl` actual fue entrenado con las 9 columnas (incluyendo `log_playcount` e `is_hit`). Para predicciones nuevas en Streamlit se derivan ambas desde los inputs del usuario:
- `log_playcount = log1p(oyentes × engagement)`
- `is_hit = 1 si playcount estimado ≥ 5.079.698`




### FEATURES finales del modelo

```python
FEATURES = [
    'log_listeners',           # Mejor predictor: canciones ya conocidas siguen siéndolo
    'duration_min',            # Duración en minutos
    'is_short_track',          # Flag formato TikTok (< 2.5 min)
    'tag_encoded',             # Género convertido a número con LabelEncoder
    'artist_track_count',      # Artistas con más tracks = más conocidos
    'track_share_of_artist',   # Si este track es el gran hit del artista
    'playcount_per_listener',  # Engagement: cuántas veces escucha cada fan
]
```



> Feature importance (importancia de cada variable)

| **Ranking** | **Feature** | **Importancia** | **Interpretación** |
|---|---|---|---|
| 1º | `log_listeners` | ~44% | Canciones populares ya tienen audiencia — la popularidad genera más popularidad |
| 2º | `tag_encoded` | ~9% | El género influye — algunos géneros tienen mucho más mercado |
| 3º | `playcount_per_listener` | ~2% | Engagement: fans que escuchan en loop son señal de hit |
| 4º | `artist_track_count` | ~0.1% | Artistas consolidados tienen más tracción |
| ... | `duration_min`, `is_short_track` | ~0% | Poca señal en las variables de duración |

* ¿Por qué `log_listeners` es tan dominante?

Porque la popularidad en música es auto-reforzante: las canciones que ya tienen muchos oyentes aparecen más en algoritmos, playlists y recomendaciones, atrayendo más oyentes. El modelo captura este patrón — las canciones que ya son conocidas en el momento de la medición tienden a seguir siendo populares.

> Por qué Random Forest

| **Criterio** | **Justificación** |
|---|---|
| Distribución de datos | Robusto con distribuciones no normales (no necesita escalar features) |
| Interpretabilidad | Feature importance explica qué variables importan más |
| Overfitting | El ensemble de árboles reduce el sobreajuste vs un árbol individual |
| Datos mixtos | Maneja variables numéricas y categóricas codificadas sin problema |

## Conclusiones y limitaciones



### Resultados del análisis



| **Análisis** | **Conclusión** |
|---|---|
| Distribución playcount | Muy sesgada → necesaria transformación log |
| Mann-Whitney | Las canciones largas tienen mayor playcount mediano (contrario a la hipótesis TikTok) |
| Kruskal-Wallis | El género influye significativamente en la popularidad (p < 0.05) |
| Feature importance | `log_listeners` es el predictor dominante |



### Limitaciones del dataset



- **Cobertura de géneros:** Solo el 10.5% de tracks tiene género asignado
- **Cobertura geográfica:** Solo el 5.8% tiene país real (el resto es GLOBAL/UNKNOWN)
- **Sin metadata de audio:** Last.fm no tiene BPM, energía, danceability (necesitaría Spotify API)
- **Data leakage en el modelo guardado:** `log_playcount` e `is_hit` incluidos como features



### Ampliaciones futuras documentadas

* Fraude en streams y Predicción a través de metadata de los Top 15 tracks del momento.